# 03_02_線形回帰をバッチと_SGD_で学習する

コード：`ch03/blocks/03_02_線形回帰をバッチと_SGD_で学習する.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 線形回帰 ŷ = a·x + b を勾配降下で学習する（バッチ／SGD）
import numpy as np

rng = np.random.default_rng(0)
x = np.linspace(0.5, 4.0, 14)                      # 14 点。0 中心ではない
y = 1.2 * x + 0.5 + rng.normal(0, 0.3, x.size)     # 正解は a=1.2, b=0.5 にノイズ

def loss(a, b):
    return np.mean((a * x + b - y) ** 2)

def grads(a, b, idx):                              # idx の点だけで勾配を計算
    r = a * x[idx] + b - y[idx]                    # 残差
    return 2 * np.mean(x[idx] * r), 2 * np.mean(r)

def train(eta, mode, steps):
    a, b, log = -0.6, 4.0, []
    pick = np.random.default_rng(1)
    for t in range(steps + 1):
        log.append((t, a, b, loss(a, b)))
        idx = np.arange(x.size) if mode == "batch" else [pick.integers(x.size)]
        ga, gb = grads(a, b, idx)
        a, b = a - eta * ga, b - eta * gb
    return log

# 損失の曲率（ヘッセ行列）から、安定な学習率の上限を出す
H = 2 * np.array([[np.mean(x * x), np.mean(x)], [np.mean(x), 1.0]])
lam = np.linalg.eigvalsh(H)
print(f"曲率 a方向/b方向 = {H[0,0]/H[1,1]:.2f}   η の上限 2/λmax = {2/lam[-1]:.4f}")
print(f"最小二乗解: a={np.polyfit(x, y, 1)[0]:.3f}  b={np.polyfit(x, y, 1)[1]:.3f}")

checkpoints = (0, 10, 50, 200, 1000)
for eta, mode in ((0.06, "batch"), (0.06, "sgd"), (0.15, "batch")):
    print(f"η={eta} {mode}")
    for t, a, b, L in train(eta, mode, 1000):
        if t in checkpoints:
            print(f"  step {t:4d}: a={a:9.3f}  b={b:9.3f}  L={L:.4g}" if L < 1e6
                  else f"  step {t:4d}: 発散（L={L:.1e}）")
